# Devoir 3 — TP marathon

Bienvenue dans votre troisième devoir ! Changement de décor : après `numpy`, nous passons à `pandas`, découvert en séance 3. Ce devoir prend la forme d'un TP guidé, dans lequel nous allons charger un jeu de données de marathons puis l'interroger : sélections, premiers agrégats, et une première rencontre avec les durées.

```{admonition} À rendre pour le dimanche 4 octobre 2026, 23 h 59
:class: important

- Ce notebook reste dans le dossier `D3/` de votre dépôt `pe-homework`, puis il doit être **commité et poussé** (`git add`, `git commit`, `git push`) avant la deadline. Attention, c'est la **date du commit** qui fait foi, pas celle du push ; le nom du fichier, lui, n'a pas d'importance (`sujet.ipynb` convient très bien).
- Juste avant de rendre, **pensez à ré-exécuter toutes les cellules** via *Kernel → Restart Kernel and Run All Cells* : toutes doivent s'exécuter dans l'ordre et sans erreur.
- Le fichier `data/marathon.txt` doit rester dans un sous-dossier `data/` à côté du notebook.
- La documentation de `pandas` et les notebooks du cours sont autorisés ; les assistants IA sont tolérés, à condition que vous soyez **capable d'expliquer chaque ligne que vous rendez** — je pourrai vous le demander à l'oral en séance.
```

```{admonition} Barème (/20)
:class: note

| Partie | Points |
|--------|--------|
| chargement propre (`sep`, `names`) + sauvegarde csv | 4 |
| recherches (1971, Londres 1981, villes) | 4 |
| extraits #1, #2, #3 (`iloc` / `loc` / masques, bons types de retour) | 5 |
| agrégats (moyenne, marathons par an) | 3 |
| durées (`to_timedelta`, `.dt`, colonnes `hour/minute/second`) | 4 |
| bonus : la section `dt.components` **et** une figure du record de Paris au fil des ans (`df.plot`) | +2 |

Les cellules « pour vérifier » doivent afficher `True` : elles jouent le rôle des cellules de test des devoirs précédents, ne les modifiez pas !
```

````{admonition} Pas encore vu en cours : df.plot (pour le bonus)
:class: note

Une `Series` ou une `DataFrame` sait se dessiner toute seule : `s.plot()` trace une courbe avec l'index en abscisse, `s.plot(marker="o")` ajoute des marqueurs, `df.plot.bar()` fait un diagramme en barres. C'est le sujet de la séance 5 ; pour le bonus, une seule ligne suffit une fois la série des records de Paris construite.
````

Ce TP est repris de <https://numerique-exos.info-mines.paris/pandas-tps/marathon/readme-marathon-nb/> (Thierry Parmentelat, CC BY-NC-ND). Le corrigé existe donc en ligne ; je compte sur vous pour ne pas aller le regarder avant d'avoir rendu !

# marathon (divers basique)

Voici un petit TP pour travailler trois briques de `pandas` :

* le chargement et la sélection ;
* un peu de `groupby` ;
* un peu de gestion du temps et des durées.

In [1]:
import pandas as pd

## les données

Nous allons étudier un jeu de données trouvé sur Internet : les temps records de plusieurs grands marathons, au fil des ans.

In [2]:
# 2024: le site original semble être *down*
# URL = "http://www.xavierdupre.fr/enseignement/complements/marathon.txt"

DATA = "data/marathon.txt"

In [3]:
# regardons les 5 premières lignes du fichier de données
# (ou bien ouvrez-le dans vs-code)

with open(DATA) as f:
    for _ in range(5):
        print(next(f), end="")

PARIS	2011	02:06:29	7589
PARIS	2010	02:06:41	7601
PARIS	2009	02:05:47	7547
PARIS	2008	02:06:40	7600
PARIS	2007	02:07:17	7637


## chargement

Pour charger un fichier de ce genre, le premier réflexe est d'utiliser la fonction `read_csv` de `pandas`.

In [4]:
# votre cellule de code
# qu'on va faire descendre
# et raffiner au fur et à mesure

df0 = pd.read_csv(DATA)
df0.head()

,PARIS\t2011\t02:06:29\t7589
0,PARIS\t2010\t02:06:41\t7601
1,PARIS\t2009\t02:05:47\t7547
2,PARIS\t2008\t02:06:40\t7600
3,PARIS\t2007\t02:07:17\t7637
4,PARIS\t2006\t02:08:03\t7683


C'est un début, mais le résultat ne ressemble pas franchement à ce qu'on espérait !

Il faut donc regarder la documentation d'un peu plus près.

In [5]:
pd.read_csv?

Signature:
pd.read_csv(
    filepath_or_buffer: 'FilePath | ReadCsvBuffer[bytes] | ReadCsvBuffer[str]',
    *,
    sep: 'str | None | lib.NoDefault' = <no_default>,
    delimiter: 'str | None | lib.NoDefault' = None,
    header: "int | Sequence[int] | None | Literal['infer']" = 'infer',
    names: 'Sequence[Hashable] | None | lib.NoDefault' = <no_default>,
    index_col: 'IndexLabel | Literal[False] | None' = None,
    usecols: 'UsecolsArgType' = None,
    dtype: 'DtypeArg | None' = None,
    engine: 'CSVEngine | None' = None,
    converters: 'Mapping[HashableT, Callable] | None' = None,
    true_values: 'list | None' = None,
    false_values: 'list | None' = None,
    skipinitialspace: 'bool' = False,
    skiprows: 'list[int] | int | Callable[[Hashable], bool] | None' = None,
    skipfooter: 'int' = 0,
    nrows: 'int | None' = None,
    na_values: 'Hashable | Iterable[Hashable] | Mapping[Hashable, Iterable[Hashable]] | None' = None,
    keep_default_na: 'bool' = True,
    na_filter: 

Pour commencer, je vous invite à préciser le séparateur :

In [6]:
# à vous de modifier cette première approche

df1 = pd.read_csv(DATA,sep="\t") #séparation avec les tabulations
df1.head()
df1.shape

(358, 4)

In [7]:
# pour vérifier, ceci doit afficher True

df1.shape == (358, 4) and df1.iloc[0, 0] == 'PARIS' and df1.columns[0] == 'PARIS'

True

C'est mieux, mais les noms des colonnes ne sont pas corrects. En effet, par défaut, `read_csv` utilise la première ligne du fichier pour déterminer les noms des colonnes ; or notre fichier texte ne contient pas de ligne d'en-tête (voyez ci-dessus) !

Il serait donc pertinent de donner nous-mêmes un nom aux colonnes.

In [8]:
NAMES = ["city", "year", "duration", "seconds"]

In [9]:
# à vous de créer une donnée bien propre
df = pd.read_csv(DATA,sep="\t",names=NAMES)

In [10]:
# pour vérifier, ceci doit afficher True

df.shape == (359, 4) and df.iloc[0, 0] == 'PARIS' and df.columns[0] == 'city'

True

In [11]:
# ce qui maintenant nous donne ceci

df.head(2)

,city,year,duration,seconds
0,PARIS,2011,02:06:29,7589
1,PARIS,2010,02:06:41,7601


## sauvegarde dans un fichier csv

Dans l'autre sens, quand on a produit une dataframe, on veut souvent sauver le résultat dans un fichier texte ; c'est le rôle de `to_csv`.

In [12]:
df.to_csv?

Signature:
df.to_csv(
    path_or_buf: 'FilePath | WriteBuffer[bytes] | WriteBuffer[str] | None' = None,
    *,
    sep: 'str' = ',',
    na_rep: 'str' = '',
    float_format: 'str | Callable | None' = None,
    columns: 'Sequence[Hashable] | None' = None,
    header: 'bool | list[str]' = True,
    index: 'bool' = True,
    index_label: 'IndexLabel | None' = None,
    mode: 'str' = 'w',
    encoding: 'str | None' = None,
    compression: 'CompressionOptions' = 'infer',
    quoting: 'int | None' = None,
    quotechar: 'str' = '"',
    lineterminator: 'str | None' = None,
    chunksize: 'int | None' = None,
    date_format: 'str | None' = None,
    doublequote: 'bool' = True,
    escapechar: 'str | None' = None,
    decimal: 'str' = '.',
    errors: 'OpenFileErrors' = 'strict',
    storage_options: 'StorageOptions | None' = None,
) -> 'str | None'
Docstring:
Write object to a comma-separated values (csv) file.

Parameters
----------
path_or_buf : str, path object, file-like object, or No

Par ex., je crée ici un fichier qu'on pourra relire sous Excel :

In [13]:
loop = "marathon-loop.csv"
df.to_csv(loop,sep=";", index=False)

In [14]:
# pour voir un aperçu
#  nouveau vous pouvez regarder le fichier avec vs-code 
# ou encore dans le terminal avec $ less marathon-loop.csv (sortir avec 'q')

%cat marathon-loop.csv

city;year;duration;seconds
PARIS;2011;02:06:29;7589
PARIS;2010;02:06:41;7601
PARIS;2009;02:05:47;7547
PARIS;2008;02:06:40;7600
PARIS;2007;02:07:17;7637
PARIS;2006;02:08:03;7683
PARIS;2005;02:08:02;7682
PARIS;2004;02:08:56;7736
PARIS;2003;02:06:33;7593
PARIS;2002;02:08:18;7698
PARIS;2001;02:09:40;7780
PARIS;2000;02:08:49;7729
PARIS;1999;02:08:10;7690
PARIS;1998;02:09:37;7777
PARIS;1997;02:10:14;7814
PARIS;1996;02:12:18;7938
PARIS;1995;02:10:06;7806
PARIS;1994;02:10:56;7856
PARIS;1993;02:10:46;7846
PARIS;1992;02:10:03;7803
PARIS;1990;02:13:10;7990
PARIS;1989;02:13:03;7983
PARIS;1988;02:13:53;8033
PARIS;1987;02:11:09;7869
PARIS;1986;02:12:44;7964
PARIS;1985;02:10:49;7849
PARIS;1984;02:11:58;7918
PARIS;1983;02:12:38;7958
PARIS;1982;02:14:07;8047
PARIS;1981;02:11:44;7904
PARIS;1980;02:25:50;8750
PARIS;1979;02:18:53;8333
PARIS;1978;02:34:55;9295
PARIS;1977;02:30:41;9041
PARIS;1976;02:20:57;8457
BERLIN;2011;02:03:38;7418
BERLIN;2010;02:05:08;7508
BERLIN;2009;02:06:08;7568
BERLIN;2008;02:03:59

## des recherches

### les éditions de 1971

In [15]:
# à vous de calculer les éditions de 1971

df_1971 = df[df['year']==1971]
df_1971.head(5)

,city,year,duration,seconds
112,FUKUOKA,1971,02:12:51,7971
204,NEW YORK,1971,02:22:54,8574
285,BOSTON,1971,02:18:45,8325


In [16]:
# ceci doit retourner True

df_1971.shape == (3, 4) and df_1971.seconds.max() == 8574

np.True_

### l'édition de 1981 à Londres

In [17]:
# à vous

df_london_1981 = df[(df['year']==1981) & (df['city']=='LONDON')]
df_london_1981

,city,year,duration,seconds
172,LONDON,1981,02:11:48,7908


In [18]:
# ceci doit retourner True

df_london_1981.shape == (1, 4) and df_london_1981.iloc[0].seconds == 7908

np.True_

### trouver toutes les villes

On veut maintenant construire une collection de toutes les villes qui apparaissent au moins une fois dans la table.

In [19]:
# à vous

cities = df['city'].unique()
print(cities)

<ArrowStringArray>
[    'PARIS',    'BERLIN',   'FUKUOKA', 'AMSTERDAM',    'LONDON',  'NEW YORK',
  'STOCKOLM',    'BOSTON',   'CHICAGO']
Length: 9, dtype: str


Intéressez-vous au type du résultat : est-ce une dataframe, une series, un ndarray, une liste ?

In [20]:
print(type(cities))

<class 'pandas.arrays.ArrowStringArray'>


C'est un pandas array de strings.

## des extraits

Attention, dans les consignes qui suivent, les numéros de ligne **commencent à 1**.

### extrait #1

Extrayez les entrées correspondant aux lignes 10 à 12 incluses.

In [21]:
# à vous

df_10_to_12 = df.iloc[9:12] #décalage des indices, la première ligne est à 0

In [22]:
# ceci doit retourner True

df_10_to_12.shape == (3, 4) and df_10_to_12.iloc[0].year == 2002 #and df_10_to_12.iloc[-1].year == 2000

np.True_

### extrait #2

Construisez une `Series` correspondant aux événements à Paris après 2000 (inclus), dans laquelle on ne garde que l'année.

In [23]:
# à vous
s_paris_2000 = df[(df['city']=='PARIS')&(df['year']>=2000)]['year']

In [24]:
s_paris_2000

0     2011
1     2010
2     2009
3     2008
4     2007
5     2006
6     2005
7     2004
8     2003
9     2002
10    2001
11    2000
Name: year, dtype: int64

In [25]:
# ceci doit retourner True

isinstance(s_paris_2000, pd.Series) and len(s_paris_2000) == 12 and s_paris_2000.iloc[-1] == 2000

np.True_

### extrait #3

Construisez maintenant une `DataFrame` correspondant aux mêmes événements — Paris après 2000 — dans laquelle on ne garde que les deux colonnes `year` et `seconds`.

In [26]:
df_paris_2000_ys = df[(df['city']=='PARIS')&(df['year']>=2000)][['year','seconds']]

df_paris_2000_ys

,year,seconds
0,2011,7589
1,2010,7601
2,2009,7547
3,2008,7600
4,2007,7637
5,2006,7683
6,2005,7682
7,2004,7736
8,2003,7593
9,2002,7698


In [27]:
# ceci doit retourner True

(isinstance(df_paris_2000_ys, pd.DataFrame)
 and df_paris_2000_ys.shape == (12, 2) 
 and df_paris_2000_ys.iloc[-2].seconds == 7780)

np.True_

## agrégats

### moyenne

Pour commencer, que vaudrait la moyenne de la colonne `seconds` ?

In [28]:
# calculer la moyenne de la colonne 'seconds'

seconds_average = df['seconds'].mean()
seconds_average

np.float64(7933.660167130919)

In [29]:
# pour vérifier

import math
math.isclose(seconds_average, 7933.660167130919)

True

````{admonition} Pas encore vu en cours : groupby
:class: note

`df.groupby("colonne")` découpe la table en groupes — un par valeur distincte de la colonne — puis applique une opération à chaque groupe : `df.groupby("city").size()` compte les lignes de chaque ville, `df.groupby("city")["seconds"].mean()` en donne la durée moyenne. Le résultat est une `Series` indexée par les valeurs du critère. On étudiera `groupby` en détail en séance 4.
````

### combien de marathons par an

Je veux maintenant produire une série qui compte, année par année, combien de marathons ont été courus.

Il y a plusieurs façons d'y arriver ; si vous en voyez plus d'une, n'hésitez pas à toutes les essayer !

In [30]:
# à vous

count_by_year = df.groupby('year').size()

In [31]:
# pour vérifier

(isinstance(count_by_year, pd.Series)
 and len(count_by_year) == 65
 and count_by_year.loc[1947] == 1
 and count_by_year.loc[2007] == 9
 and count_by_year.loc[2011] == 5)

np.True_

## les durées

Dans cette partie, notre but est simplement de vérifier que la colonne `seconds` contient bien le nombre de secondes correspondant à la colonne `duration`.

Pour ce faire, nous allons commencer par convertir la colonne `duration` en quelque chose d'un peu plus utilisable. En effet, `numpy` expose deux types particulièrement bien adaptés à la gestion du temps :

* `datetime64` pour modéliser un instant particulier ;
* `timedelta64` pour modéliser une durée entre deux instants.

Au besoin, vous trouverez plus de détails ici : <https://numpy.org/doc/stable/reference/arrays.datetime.html>

### `read_csv(parse_dates=)`

Commençons par écarter une "fausse bonne idée" : `read_csv` propose une option `parse_dates`. Regardez ce que cela donne ici :

In [32]:
df_broken = pd.read_csv(
    DATA, sep='\t', 
    names=['city', 'year', 'duration', 'seconds'], 
    parse_dates=['duration'])
df_broken

/tmp/ipykernel_42936/3523436968.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_broken = pd.read_csv(


,city,year,duration,seconds
0,PARIS,2011,2026-09-29 02:06:29,7589
1,PARIS,2010,2026-09-29 02:06:41,7601
2,PARIS,2009,2026-09-29 02:05:47,7547
3,PARIS,2008,2026-09-29 02:06:40,7600
4,PARIS,2007,2026-09-29 02:07:17,7637
...,...,...,...,...
354,CHICAGO,2006,2026-09-29 02:07:35,7655
355,CHICAGO,2007,2026-09-29 02:11:11,7871
356,CHICAGO,2008,2026-09-29 02:06:25,7585
357,CHICAGO,2009,2026-09-29 02:05:41,7541


Ça ne va pas du tout ! Pouvez-vous deviner pourquoi ? Le souci, c'est qu'ici nous n'avons pas une *date* : ce que nous avons, c'est une *durée* — et ce n'est pas du tout la même chose.

````{admonition} Pas encore vu en cours : pd.to_timedelta et l'accesseur .dt
:class: note

`pd.to_timedelta(serie)` convertit une colonne de textes comme `"2:06:29"` en vraies durées (type `timedelta64`), sur lesquelles additions, soustractions et comparaisons ont un sens. Une fois la colonne convertie, l'accesseur `.dt` donne accès aux méthodes propres aux durées, exactement comme `.str` pour les chaînes de caractères : `serie.dt.total_seconds()`. Les dates et les durées sont le sujet de la séance 5.
````

### `pd.to_timedelta()`

In [33]:
# repartons des données de départ

df = pd.read_csv(DATA, sep="\t", names=NAMES)

df.dtypes

city          str
year        int64
duration      str
seconds     int64
dtype: object

Pour convertir proprement la colonne, nous allons plutôt utiliser `pd.to_timedelta()`.

Voyez la documentation de cette fonction, puis modifiez la dataframe `df` pour que la colonne `duration` soit désormais du type `timedelta64`.

In [34]:
# à vous
df['duration']=pd.to_timedelta(df['duration'])
df.dtypes

city                    str
year                  int64
duration    timedelta64[us]
seconds               int64
dtype: object

In [35]:
# pour vérifier - doit retourner True

df.duration.dtype.kind == 'm'   # timedelta64 (ns ou us selon la version de pandas)

True

In [36]:
# et maintenant ça devrait être beaucoup mieux

df.head(2)

,city,year,duration,seconds
0,PARIS,2011,0 days 02:06:29,7589
1,PARIS,2010,0 days 02:06:41,7601


In [37]:
# une fois que vous avez bien converti vous pourrez faire ceci
df.duration.dt.components

,days,hours,minutes,seconds,milliseconds,microseconds,nanoseconds
0,0,2,6,29,0,0,0
1,0,2,6,41,0,0,0
2,0,2,5,47,0,0,0
3,0,2,6,40,0,0,0
4,0,2,7,17,0,0,0
...,...,...,...,...,...,...,...
354,0,2,7,35,0,0,0
355,0,2,11,11,0,0,0
356,0,2,6,25,0,0,0
357,0,2,5,41,0,0,0


### duration == seconds ?

À présent que `duration` est dans le bon type, nous pouvons utiliser toutes les fonctions disponibles sur ce type. À l'usage, cela se fait en deux temps :

* sur l'objet `Series`, on applique l'accesseur `.dt` pour, en quelque sorte, se projeter dans l'espace des durées ; c'est exactement le même mécanisme que le `.str` que nous avons utilisé pour appliquer des méthodes comme `.lower()` ou `.replace()` aux chaînes, et non à la série (plus de détails ici : <https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.html>)
* de là, on peut appeler toutes les méthodes disponibles sur les objets `timedelta` ; intéressez-vous en particulier à `total_seconds`.

Ainsi, pour vérifier que la colonne `seconds` correspond bien à `duration`, quel code écririez-vous ? (il doit afficher `True`)

In [38]:
(df["duration"].dt.total_seconds()==df['seconds']).all()

np.True_

### colonnes `hour` `minute` et `second`

Nous nous proposons maintenant d'ajouter des colonnes `hour`, `minute` et `second`, qui doivent être de type entier. Pour ce faire, deux approches :

- "à la main" : on fait les calculs nous-mêmes ;
- puis la version "paresseuse" : on découvre au détour d'une question StackOverflow que l'information est disponible directement dans la colonne `duration` — mais elle est bien cachée !

#### à la main

Un indice : on peut calculer le quotient et le reste entre deux objets de type "durée" avec les opérateurs usuels `//` et `%`, comme illustré ci-dessous.

In [39]:
# par exemple
import numpy as np

# une durée de 1h
one_hour = np.timedelta64(1, 'h')
# guess what...
one_minute = np.timedelta64(1, 'm')
one_second = np.timedelta64(1, 's')

# une durée de 2h25
random_duration = 2*one_hour + np.timedelta64(25, 'm')


# eh bien on peut faire comme avec des entiers

quotient, reste = random_duration // one_hour, random_duration % one_hour

quotient, reste

(np.int64(2), np.timedelta64(25,'m'))

Maintenant que nous savons faire tout cela, à vous de calculer les colonnes `hour`, `minute` et `second`.

In [40]:
df['hour']=df['duration']//one_hour
df['minute']=df['duration']%one_hour
df['second']=(df['minute']%one_minute)//one_second # %renvoie une durée, contrairement à //
df['minute']=df['minute']//one_minute


In [41]:
# pour vérifier, vous décommentez tout ceci et ça doit afficher True
(    np.all(df.loc[0, ['hour', 'minute', 'second']] == [2, 6, 29])
  and df.hour.dtype == int
  and df.minute.dtype == int 
  and df.second.dtype == int)

True

#### version paresseuse avec `dt.components`

Il se trouve qu'on peut faire le même travail sans s'embêter autant, une fois qu'on a découvert que [l'accesseur `.dt` possède un attribut qui donne accès à ce genre de détails](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.dt.components.html).

In [42]:
# on défait le travail de la section précédente, si nécessaire

for col in 'hour', 'minute', 'second':
    if col in df.columns:
        df.drop(columns=col, inplace=True)

In [43]:
df['hour']=df.duration.dt.components['hours']
df['minute']=df.duration.dt.components['minutes']
df['second']=df.duration.dt.components['seconds']

In [44]:
# pour vérifier: même consigne
(    np.all(df.loc[0, ['hour', 'minute', 'second']] == [2, 6, 29])
  and df.hour.dtype == int
  and df.minute.dtype == int 
  and df.second.dtype == int)

True

Bon courage à tous ! Pour toute question sur le sujet, vous pouvez me contacter par email : [raphael@imagine-app.fr](mailto:raphael@imagine-app.fr).

Au plaisir de vous lire,

votre responsable de cours, Raphael Prasquier
(sujet conçu par Aurélien Noce)